<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-08-gradio-rag-chatbot/notebooks/Pfizer_Document_QA_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Q&A Chatbot
**Pfizer AI Document Intelligence Externship — Gradio chatbot interface**

Upload one or more PDFs and ask questions about them. The bot retrieves the passages that
actually answer the question and writes an answer from those passages only, with page
citations underneath so every claim can be checked.

**Beyond the tutorial version, this one adds:**

| feature | where |
|---|---|
| multiple PDF upload | `gr.File(file_count="multiple")` |
| answer-style dropdown (short / detailed / bullets) | `mode` dropdown feeding the prompt |
| search scope dropdown (all files or one) | `doc_choice` dropdown filtering retrieval |
| live status while it works | generator that yields "Searching…" then "Writing…" |
| page-level source citations | appended under every answer |
| download the transcript as Markdown | "Save transcript" button |
| custom theme, title and emoji | `gr.themes.Soft(primary_hue="teal")` |

Retrieval: `BAAI/bge-small-en-v1.5`. Generation picks itself to match the runtime:
Mistral-7B-Instruct-v0.3 in 4-bit when a GPU is available, Qwen2.5-1.5B-Instruct on CPU when
it is not, so the notebook runs end to end either way and never needs an API key.

> A T4 GPU gives much better answers and much faster ones. If Colab has no GPU free, the
> notebook falls back automatically rather than failing.

## 1. Setup

In [ ]:
!pip -q install gradio pymupdf sentence-transformers "transformers>=4.44" accelerate bitsandbytes

import re, json, datetime
import numpy as np
import torch, pymupdf, gradio as gr
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("gradio", gr.__version__, "| CUDA", torch.cuda.is_available())

gradio 6.26.0 | CUDA False


In [ ]:
# Retrieval model, plus a chat model chosen to match whatever runtime is available.
EMBED_ID  = "BAAI/bge-small-en-v1.5"
GPU_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"   # 4-bit Mistral-7B when a GPU is free
CPU_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"                  # fallback so the notebook still runs without a GPU

USE_GPU = torch.cuda.is_available()
MODEL_ID = GPU_MODEL if USE_GPU else CPU_MODEL
MAX_NEW_TOKENS = 320 if USE_GPU else 160                  # CPU generation is slower, so ask for less

embedder = SentenceTransformer(EMBED_ID, device="cuda" if USE_GPU else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if USE_GPU:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                                 device_map="auto", torch_dtype=torch.float16)
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
model.eval()

print(f"running on {'GPU' if USE_GPU else 'CPU'} with {MODEL_ID}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

running on CPU with Qwen/Qwen2.5-1.5B-Instruct


## 2. Reading and indexing the PDFs

Every chunk remembers which file and which page it came from. That is what makes the
citations under each answer possible.

In [ ]:
CHUNK, OVERLAP = 700, 120     # characters; overlap stops a sentence being cut in half

def pdf_chunks(path, filename):
    """Split one PDF into overlapping chunks, keeping file name and page number on each."""
    doc = pymupdf.open(path)
    pieces = []
    for pno, page in enumerate(doc):
        text = re.sub(r"\n{3,}", "\n\n", page.get_text("text").strip())
        if len(text) < 20:                    # skip blank or image-only pages
            continue
        start = 0
        while start < len(text):
            pieces.append({"text": text[start:start + CHUNK], "file": filename, "page": pno})
            if start + CHUNK >= len(text):
                break
            start += CHUNK - OVERLAP
    return pieces


def build_index(files):
    """Read every uploaded PDF, chunk it and embed the chunks. Returns (state, status, choices)."""
    if not files:
        return None, "No file loaded yet.", gr.update(choices=["All documents"], value="All documents")

    chunks = []
    for f in files:
        path = f.name if hasattr(f, "name") else f
        name = path.split("/")[-1]
        try:
            chunks.extend(pdf_chunks(path, name))
        except Exception as e:
            return None, f"Could not read {name}: {type(e).__name__}", gr.update()

    if not chunks:
        return None, "Those PDFs had no extractable text. A scanned PDF would need OCR first.", gr.update()

    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                              batch_size=32, show_progress_bar=False)
    names = sorted({c["file"] for c in chunks})
    pages = len({(c["file"], c["page"]) for c in chunks})
    state = {"chunks": chunks, "vectors": np.asarray(vectors), "files": names}
    status = f"Indexed {len(names)} file(s), {pages} pages, {len(chunks)} chunks. Ask me anything."
    return state, status, gr.update(choices=["All documents"] + names, value="All documents")


def retrieve(question, state, doc_choice, k=4):
    """Cosine similarity over the chunk vectors, optionally restricted to one file."""
    q = embedder.encode([question], normalize_embeddings=True)[0]
    scores = state["vectors"] @ q
    order = np.argsort(-scores)
    hits = []
    for i in order:
        c = state["chunks"][i]
        if doc_choice and doc_choice != "All documents" and c["file"] != doc_choice:
            continue
        hits.append({**c, "score": float(scores[i])})
        if len(hits) == k:
            break
    return hits


# A sample pharmaceutical PDF so the app can be demoed without uploading anything by hand.
import os, gdown
SAMPLE = "pharma-blob-sample.pdf"
if not os.path.exists(SAMPLE):
    gdown.download(id="1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw", output=SAMPLE, quiet=True)

def load_sample():
    """Index the bundled sample document."""
    return build_index([SAMPLE])

## 3. Writing the answer

The dropdown changes one line of the prompt, so the same retrieved passages can come back as
a one-liner, a full explanation or a bullet list. The bot is told to say when the passages do
not contain the answer, which is what stops it inventing one.

In [ ]:
STYLE = {
    "Short answer":     "Answer in one or two sentences. No preamble.",
    "Detailed answer":  "Answer in a short paragraph, quoting the exact figures, dates and identifiers.",
    "Bullet summary":   "Answer as 3 to 5 short bullet points, one fact per bullet.",
}

@torch.inference_mode()
def generate_answer(question, hits, mode):
    """Build a grounded prompt from the retrieved passages and generate one answer."""
    context = "\n\n".join(
        f"[{i+1}] ({h['file']}, page {h['page']+1})\n{h['text']}" for i, h in enumerate(hits))
    prompt = (
        "You are a careful assistant answering questions about an uploaded document.\n"
        "Use ONLY the passages below. If they do not contain the answer, say so plainly "
        "instead of guessing.\n"
        f"{STYLE.get(mode, STYLE['Short answer'])}\n\n"
        f"PASSAGES:\n{context}\n\nQUESTION: {question}\n\nANSWER:"
    )
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         temperature=None, top_p=None,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def cite(hits):
    """One line of page-level sources under the answer."""
    seen = sorted({f"{h['file']} p.{h['page']+1}" for h in hits})
    return "\n\n`sources: " + " · ".join(seen) + "`"

## 4. The chat handler

`respond` is a **generator**: it yields the user's message straight away, then a status line
while it searches, then the finished answer. That is what gives the interface a live
"working on it" state instead of a frozen button.

In [ ]:
def respond(message, history, mode, doc_choice, state):
    history = list(history or [])
    message = (message or "").strip()
    if not message:
        yield history, "", "Type a question first."
        return

    history.append({"role": "user", "content": message})

    if not state or not state.get("chunks"):
        history.append({"role": "assistant",
                        "content": "Upload a PDF first and I'll answer questions about it."})
        yield history, "", "No document loaded yet."
        return

    yield history, "", "Searching the document..."        # status update 1
    hits = retrieve(message, state, doc_choice, k=4)

    if not hits:
        history.append({"role": "assistant", "content": "Nothing in that file matched the question."})
        yield history, "", "No matching passages."
        return

    yield history, "", f"Found {len(hits)} passages. Writing the answer..."   # status update 2
    answer = generate_answer(message, hits, mode)
    history.append({"role": "assistant", "content": answer + cite(hits)})
    yield history, "", "Ready."


def save_transcript(history):
    """Write the conversation to a Markdown file the user can download."""
    lines = ["# Document Q&A transcript", f"_{datetime.datetime.now():%Y-%m-%d %H:%M}_", ""]
    for m in history or []:
        who = "**You**" if m["role"] == "user" else "**Bot**"
        lines.append(f"{who}: {m['content']}\n")
    path = "chat_transcript.md"
    with open(path, "w") as f:
        f.write("\n".join(lines))
    return gr.update(value=path, visible=True)


def clear_chat():
    return [], "", "Chat cleared.", gr.update(visible=False)

## 5. The interface

In [ ]:
THEME = gr.themes.Soft(primary_hue="teal", secondary_hue="slate", neutral_hue="slate")

with gr.Blocks() as demo:
    state = gr.State(None)          # holds chunks + vectors between clicks

    gr.Markdown(
        "# 📄 Document Q&A Chatbot\n"
        "Upload one or more PDFs on the left, then ask about them. "
        "Every answer cites the page it came from."
    )

    with gr.Row():
        # ---------------- left column: documents and controls ----------------
        with gr.Column(scale=1):
            files = gr.File(label="📎 Upload PDFs", file_count="multiple", file_types=[".pdf"])
            load_btn = gr.Button("Index documents", variant="primary")
            sample_btn = gr.Button("📥 Load sample PDF")
            status = gr.Markdown("No document loaded yet.")

            mode = gr.Dropdown(list(STYLE), value="Short answer", label="🎚️ Answer style")
            doc_choice = gr.Dropdown(["All documents"], value="All documents",
                                     label="🔎 Search in")

            with gr.Row():
                save_btn = gr.Button("💾 Save transcript")
                clear_btn = gr.Button("🧹 Clear chat")
            transcript = gr.File(label="Your transcript", visible=False)

        # ---------------- right column: the conversation ----------------
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Conversation", height=430,   # Gradio 6 uses the messages format by default
                                 avatar_images=(None, "https://em-content.zobj.net/source/twitter/376/robot_1f916.png"))
            msg = gr.Textbox(placeholder="Ask something about the document, then press Enter",
                             label="Your question", lines=2)
            with gr.Row():
                send = gr.Button("Send", variant="primary")
            gr.Examples(
                label="Try one of these",
                examples=["What documents are included in this package?",
                          "Which lot numbers are listed and did they pass their tests?",
                          "Does the product contain any material of animal origin?",
                          "Were there any packaging configuration changes?"],
                inputs=msg,
            )

    # ---------------- wiring ----------------
    load_btn.click(build_index, inputs=files, outputs=[state, status, doc_choice])
    sample_btn.click(load_sample, outputs=[state, status, doc_choice])
    files.upload(build_index, inputs=files, outputs=[state, status, doc_choice])

    chat_inputs = [msg, chatbot, mode, doc_choice, state]
    chat_outputs = [chatbot, msg, status]
    msg.submit(respond, chat_inputs, chat_outputs)
    send.click(respond, chat_inputs, chat_outputs)

    save_btn.click(save_transcript, inputs=chatbot, outputs=transcript)
    clear_btn.click(clear_chat, outputs=[chatbot, msg, status, transcript])

demo.queue()

# Gradio 6 moved theme and title onto launch(); the fallback keeps older versions working.
try:
    demo.launch(share=True, theme=THEME, title="Document Q&A Chatbot")
except TypeError:
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1f5b0f081aae32d22f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# The public link, printed on its own so it is easy to copy
print("public app link:", demo.share_url)

public app link: https://1f5b0f081aae32d22f.gradio.live


## 6. What changed compared to the tutorial

The tutorial version was a single chat box wired to a model. This one is a small application:

- **It reads real documents.** PDFs are chunked with overlap and embedded, and the answer is
  written only from the passages retrieved for that specific question.
- **It shows its work.** Every answer ends with the file and page each passage came from, so
  an answer can be checked rather than trusted.
- **It has a two-column layout** instead of one stacked column: documents and controls on the
  left, conversation on the right.
- **Two dropdowns change behaviour without touching code** — answer style rewrites one line of
  the prompt, search scope restricts retrieval to a single file.
- **The handler is a generator**, so the status line moves through "Searching" and "Writing"
  rather than the UI sitting frozen.
- **The transcript can be downloaded** as Markdown.
- **It refuses rather than guesses.** The prompt tells it to say when the passages do not
  contain the answer, which matters far more on regulated documents than a confident tone does.